# Full-data entity-resolution audit

This notebook reads E00 evidence from the supplied TSVs. Source grain: one record per source entity; truth grain: one S1 entity and its target-ID list. No external data or identity lookup. Run the reproduction cell only when no pipeline process holds the cache.


In [ ]:
from pathlib import Path
import json, sys
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
audit = json.loads((root / 'reports/data_audit.json').read_text(encoding='utf-8'))
for name, info in audit['sources'].items():
    print(name, 'rows:', info['rows'], 'duplicate IDs:', info['duplicate_ids'])
    print('Countries:', info['countries'])
    print('Missing addresses:', info['business_address_missing'])
    print('S1 raw collisions:', info.get('collisions', {}))


In [ ]:
print(json.dumps(audit['integrity'], indent=2))
print(json.dumps(audit['truth_match_counts'], indent=2))
print(json.dumps(audit['truth_source_mix'], indent=2))


## Inspect the exact SQL and reproduce

The module below contains all schema, uniqueness, missingness, length/token, country, collision and label-reference queries. A full run reads all 24.23 million source records. Counts are exact; sample noise tags are heuristic. No dates are supplied for temporal drift checks.


In [ ]:
import inspect
from src.audit import audit as run_audit
print(inspect.getsource(run_audit))


In [ ]:
# Explicit full-data rerun; output overwrites the audit report with a fresh run.
run_audit(root)


## Interpretation

No duplicate source IDs, broken truth references, shared secondary ownership or cross-split ID overlaps were found. Missing secondary addresses and extensive S1 name collisions make name-only decisions unsafe. Singletons require explicit empty predictions. France has no training labels, so its quality cannot be inferred from in-country training metrics. See `reports/data_audit.md` and the positive-pair sample for evidence.
